In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
import re
import joblib
import string
import nltk
from sklearn.feature_extraction.text import TfidfVectorizer
from nltk.corpus import stopwords   

In [5]:
fake=pd.read_csv('Fake.csv')
true=pd.read_csv('True.csv')
fake.head()

,title,text,subject,date
0,Donald Trump Sends Out Embarrassing New Year’...,Donald Trump just couldn t wish all Americans ...,News,"December 31, 2017"
1,Drunk Bragging Trump Staffer Started Russian ...,House Intelligence Committee Chairman Devin Nu...,News,"December 31, 2017"
2,Sheriff David Clarke Becomes An Internet Joke...,"On Friday, it was revealed that former Milwauk...",News,"December 30, 2017"
3,Trump Is So Obsessed He Even Has Obama’s Name...,"On Christmas day, Donald Trump announced that ...",News,"December 29, 2017"
4,Pope Francis Just Called Out Donald Trump Dur...,Pope Francis used his annual Christmas Day mes...,News,"December 25, 2017"


In [6]:
true.head()

,title,text,subject,date
0,"As U.S. budget fight looms, Republicans flip t...",WASHINGTON (Reuters) - The head of a conservat...,politicsNews,"December 31, 2017"
1,U.S. military to accept transgender recruits o...,WASHINGTON (Reuters) - Transgender people will...,politicsNews,"December 29, 2017"
2,Senior U.S. Republican senator: 'Let Mr. Muell...,WASHINGTON (Reuters) - The special counsel inv...,politicsNews,"December 31, 2017"
3,FBI Russia probe helped by Australian diplomat...,WASHINGTON (Reuters) - Trump campaign adviser ...,politicsNews,"December 30, 2017"
4,Trump wants Postal Service to charge 'much mor...,SEATTLE/WASHINGTON (Reuters) - President Donal...,politicsNews,"December 29, 2017"


In [11]:
fake.isnull().sum()
true.isnull().sum()
true.shape
fake.shape

(23481, 4)

In [12]:
fake["class"]=0
true["class"]=1

In [14]:
fake.head()
true.head()

,title,text,subject,date,class
0,"As U.S. budget fight looms, Republicans flip t...",WASHINGTON (Reuters) - The head of a conservat...,politicsNews,"December 31, 2017",1
1,U.S. military to accept transgender recruits o...,WASHINGTON (Reuters) - Transgender people will...,politicsNews,"December 29, 2017",1
2,Senior U.S. Republican senator: 'Let Mr. Muell...,WASHINGTON (Reuters) - The special counsel inv...,politicsNews,"December 31, 2017",1
3,FBI Russia probe helped by Australian diplomat...,WASHINGTON (Reuters) - Trump campaign adviser ...,politicsNews,"December 30, 2017",1
4,Trump wants Postal Service to charge 'much mor...,SEATTLE/WASHINGTON (Reuters) - President Donal...,politicsNews,"December 29, 2017",1


In [15]:
data=pd.concat([fake,true],axis=0)

In [16]:
data.sample(10)

,title,text,subject,date,class
6477,White House says it did not leak material used...,WASHINGTON (Reuters) - The White House said on...,politicsNews,"January 6, 2017",1
11620,British PM May says Russia trying to weaponize...,WARSAW (Reuters) - British Prime Minister Ther...,worldnews,"December 21, 2017",1
11807,Kidnapped aid workers in South Sudan released:...,JUBA (Reuters) - Six aid workers kidnapped by ...,worldnews,"December 20, 2017",1
13105,U.N. rights boss urges Mexico not to enshrine ...,GENEVA (Reuters) - The United Nations human ri...,worldnews,"December 5, 2017",1
574,"Trudeau Does What Trump Can’t, Calls Out Whit...",It s no secret that Donald Trump courts white ...,News,"August 13, 2017",0
11013,LOL! CROWD CHANTS “CNN Sucks” At Trump Rally W...,President Trump skipped the White House Corres...,politics,"Apr 30, 2017",0
1048,Trump Tweets Lie About America’s Third Bigges...,Donald Trump has been tweeting all day and h...,News,"June 22, 2017",0
17071,You Won’t Believe Why Asylum Was Given To Over...,We re giving asylum to people who re connected...,Government News,"Sep 29, 2015",0
7008,North Carolina Republican Throws Temper Tantr...,In retaliation of Bruce Springsteen cancelling...,News,"April 9, 2016",0
18219,"Russia, Saudi Arabia cement new friendship wit...",MOSCOW (Reuters) - Russian President Vladimir ...,worldnews,"October 5, 2017",1


In [23]:
cols_to_drop = ["title", "subject", "date"]
data = data.drop(columns=[c for c in cols_to_drop if c in data.columns])
data = data.reset_index(drop=True)

In [24]:
data.sample(5)

,text,class
229,It s pretty clear that Donald Trump has no ide...,0
39405,LONDON (Reuters) - British foreign minister Bo...,1
38479,NEW YORK/SINGAPORE (Reuters) - United Airlines...,1
31394,"LONDON, Oct 6 (Reuters) - Officials from the U...",1
16297,Leading Republicans have already expressed an...,0


In [ ]:
def clean_text(text):
    text=text.lower();
    text=re.sub('\[.*?\]','',text)
    text=re.sub("\\W"," ",text)
    text=re.sub('https?://\S+|www\.\S+','',text)
    text=re.sub('<.*?>+','',text)   
    text=re.sub('[%s]' % re.escape(string.punctuation),'',text)
    text=re.sub('\n','',text)
    text=re.sub("\w*\d\w*","",text)
    return text


In [26]:
data["text"]=data["text"].apply(clean_text)

In [27]:
data.sample(5)

,text,class
21987,washington dc this week the senate intellige...,0
44307,paris reuters france s overseas territorie...,1
41643,tokyo reuters tokyo governor yuriko koike ...,1
19343,a highly anticipated declassified us intellige...,0
33436,washington reuters the u s congress was i...,1


In [37]:
X=data["text"]
y=data["class"]

X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.25,random_state=42)

In [38]:
# Vectorization
vectorizer=TfidfVectorizer()
X_train=vectorizer.fit_transform(X_train)
X_test=vectorizer.transform(X_test)


In [40]:
lr=LogisticRegression()
lr.fit(X_train,y_train)

LogisticRegression()

In [42]:
predictions=lr.predict(X_test)
lr.score(X_test,y_test)
# print(classification_report(y_test,predictions))


0.9858351893095768

In [ ]:
print(classification_report(y_test,predictions))

              precision    recall  f1-score   support

           0       0.99      0.99      0.99      5895
           1       0.98      0.99      0.99      5330

    accuracy                           0.99     11225
   macro avg       0.99      0.99      0.99     11225
weighted avg       0.99      0.99      0.99     11225



In [46]:
joblib.dump(vectorizer,'vectorizer.jb')
joblib.dump(lr,"lr_model.jb")

['lr_model.jb']